---
tags: [javascript, oop, constructors, error-handling]
---

# Functional Constructors & Errors in JavaScript

Constructor functions and the `Error` hierarchy are two places where JavaScript's prototype system shows through most clearly. Errors are themselves constructed objects, so understanding one explains the other.

---

## 1. Functional Constructors

A constructor function is an ordinary function invoked with `new` to act as a blueprint for objects. Convention: capitalise the name.

```js
function User(name, role) {
  // 'this' is implicitly created as a new object: {}
  this.name = name;
  this.role = role;
  // 'this' is implicitly returned
}

const admin = new User('Alice', 'Admin');
```

### What `new` actually does

Four steps, in order:

1. Creates a fresh empty object.
2. Sets that object's `[[Prototype]]` to `User.prototype`.
3. Calls `User` with `this` bound to the new object.
4. Returns the new object — **unless** the function explicitly returns an object.

That last clause is the part people miss:

```js
function A() { this.x = 1; return { x: 99 }; }
function B() { this.x = 1; return 42; }

new A().x;  // 99  — object return overrides 'this'
new B().x;  // 1   — primitive return is ignored
```

### Methods belong on the prototype, not in the constructor

```js
// Bad: a new function object allocated per instance
function User(name) {
  this.name = name;
  this.greet = function () { return `Hi ${this.name}`; };
}

// Good: one shared function on the prototype
function User(name) { this.name = name; }
User.prototype.greet = function () { return `Hi ${this.name}`; };
```

Instance properties go in the constructor; behaviour goes on the prototype. The prototype method is looked up through the chain, so it costs nothing per instance.

### The missing-`new` trap

Called without `new`, `this` is the global object (non-strict) or `undefined` (strict / modules). Either way it fails — silently polluting globals in the first case, throwing `TypeError: Cannot set property of undefined` in the second.

```js
const u = User('Alice');  // u === undefined, and window.name got clobbered
```

### Safeguarding with `new.target`

`new.target` is the constructor that `new` was called on, or `undefined` for a plain call.

```js
function User(name) {
  if (!new.target) {
    return new User(name);  // auto-corrects a missing 'new'
  }
  this.name = name;
}
```

Or throw instead of auto-correcting, if you'd rather surface the mistake:

```js
if (!new.target) throw new TypeError("User must be called with 'new'");
```

ES6 `class` does this guard for you automatically — calling a class without `new` always throws.

### Identity checks

```js
admin instanceof User;                        // true
Object.getPrototypeOf(admin) === User.prototype; // true — the real test
admin.constructor === User;                   // true, but easy to break
```

`instanceof` walks the prototype chain. It is **not** reliable across realms (iframes, Node `vm` contexts, worker boundaries) because each realm has its own `User` and its own `Error`.

---

## 2. Built-in Error Constructors

Every error thrown at runtime is an object built by one of these.

| Constructor | Thrown when |
|---|---|
| `Error` | Generic base; also what you use for custom throws |
| `TypeError` | Value is not of the expected type (`undefined is not a function`) |
| `ReferenceError` | Dereferencing an invalid reference (undeclared variable, TDZ access) |
| `SyntaxError` | Code cannot be parsed (also thrown by `JSON.parse`) |
| `RangeError` | Numeric value outside its valid range (`new Array(-1)`, stack overflow) |
| `URIError` | Malformed URI passed to `decodeURIComponent` etc. |
| `EvalError` | Legacy; no longer thrown by modern engines |
| `AggregateError` | Multiple errors at once — produced by `Promise.any` when all reject |

All of them inherit from `Error.prototype`, so `err instanceof Error` is true for every one.

### Anatomy of an error instance

- **`message`** — human-readable description passed to the constructor.
- **`name`** — the type name, e.g. `"TypeError"`. Lives on the prototype by default.
- **`stack`** — non-standard but universally implemented stack trace string.
- **`cause`** — optional (ES2022), used to chain the underlying error.

```js
try {
  throw new TypeError("Invalid input type", { cause: "Expected Array" });
} catch (err) {
  console.log(err.name);     // "TypeError"
  console.log(err.message);  // "Invalid input type"
  console.log(err.cause);    // "Expected Array"
  console.log(String(err));  // "TypeError: Invalid input type"
}
```

`cause` is most useful holding the *original error*, not a string — it preserves the lower-level stack while you rethrow something meaningful to the caller:

```js
try {
  await db.query(sql);
} catch (err) {
  throw new Error('Failed to load user profile', { cause: err });
}
```

### `try` / `catch` / `finally`

```js
try {
  risky();
} catch {              // binding is optional since ES2019
  cleanup();
} finally {
  alwaysRuns();        // runs on success, on throw, and on early return
}
```

Two things worth knowing about `finally`:

- It runs even when `try` or `catch` returns — the return value is computed, then `finally` executes, then the function returns.
- A `return` or `throw` **inside** `finally` overrides whatever `try`/`catch` was returning or throwing. That silently swallows errors, so avoid it.

---

## 3. Custom Errors via Functional Constructors

For domain-specific errors — `ValidationError`, `DatabaseError` — inherit from the native `Error`.

```js
function ValidationError(message) {
  // 1. Initialise properties
  this.name = 'ValidationError';
  this.message = message || 'Validation failed';

  // 2. Capture the stack trace cleanly (V8-specific)
  if (Error.captureStackTrace) {
    Error.captureStackTrace(this, ValidationError);
  } else {
    this.stack = new Error().stack;
  }
}

// 3. Inherit from Error's prototype
ValidationError.prototype = Object.create(Error.prototype);
ValidationError.prototype.constructor = ValidationError;

try {
  throw new ValidationError("Age must be above 18");
} catch (error) {
  console.log(error instanceof ValidationError); // true
  console.log(error instanceof Error);           // true
  console.log(error.stack);                      // clean stack trace
}
```

The second argument to `Error.captureStackTrace` omits the constructor's own frame from the trace, so the stack starts at the line that threw rather than inside `ValidationError`.

Note that `Object.create(Error.prototype)` **replaces** the prototype object, which destroys the default `constructor` link — hence step 3's second line.

---

## 4. Modern Class Syntax

```js
class ValidationError extends Error {
  constructor(message, options) {
    super(message, options);   // sets message and cause
    this.name = 'ValidationError';
  }
}

class FieldError extends ValidationError {
  constructor(field, message) {
    super(message);
    this.name = 'FieldError';
    this.field = field;
  }
}
```

`super(message)` handles message assignment and stack capture. Subclassing further is trivial, which is the main practical argument for classes here.

### Constructors vs. classes

| | Function constructor | `class` |
|---|---|---|
| Hoisting | Fully hoisted, callable before definition | Hoisted but in TDZ — throws if used early |
| Called without `new` | Silently misbehaves | Always throws `TypeError` |
| Strict mode | Inherits surrounding mode | Body always strict |
| Inheritance | Manual `Object.create` + `constructor` fixup | `extends` + `super` |
| Methods | Enumerable by default | Non-enumerable, skipped by `for...in` |
| Under the hood | — | Still prototypes; syntactic sugar |

Classes are not a different object model. `typeof ValidationError === 'function'` either way.

> **Transpilation caveat:** if Babel/TypeScript targets ES5, `extends Error` breaks `instanceof` because ES5 constructors can't properly subclass built-ins. Fix with `Object.setPrototypeOf(this, new.target.prototype)` after `super()`, or target ES6+.

---

## 5. Gotchas

**Throw `Error` objects, never strings.** `throw 'oops'` gives you no stack, no name, and `err.message` is `undefined`. Since anything can be thrown, defensive code sometimes needs `err instanceof Error ? err.message : String(err)`.

**`JSON.stringify(err)` returns `{}`.** `message` and `stack` are non-enumerable own properties. For logging, build the object explicitly:

```js
JSON.stringify({ name: err.name, message: err.message, stack: err.stack });
```

**`try/catch` does not cross async boundaries.** This catches nothing:

```js
try {
  setTimeout(() => { throw new Error('boom'); }, 0);
} catch (e) { /* never runs */ }
```

The callback executes on a later tick, with the `try` block long gone. Same for `.then()` callbacks — use `.catch()`, or `await` inside the `try`:

```js
try {
  await somethingAsync();   // this DOES work
} catch (e) { handle(e); }
```

**Prefer narrow catches.** `catch (e) { }` swallows typos and `ReferenceError`s along with the error you expected. Check the type and rethrow what you don't own:

```js
catch (err) {
  if (!(err instanceof ValidationError)) throw err;
  showFieldMessage(err.message);
}
```

**Node adds `err.code`.** Filesystem and network errors carry stable string codes (`ENOENT`, `ECONNREFUSED`) that are safer to branch on than message text, which changes between versions.

**Last-resort handlers:**
- Browser: `window.onerror`, `window.addEventListener('unhandledrejection', ...)`
- Node: `process.on('uncaughtException')`, `process.on('unhandledRejection')`

Use these for logging and graceful shutdown, not for control flow.

---

## References

- [MDN — Error](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Reference/Global_Objects/Error)
- [MDN — new operator](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Reference/Operators/new)
- [MDN — new.target](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Reference/Operators/new.target)
- [MDN — Classes](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Reference/Classes)
- [MDN — Inheritance and the prototype chain](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Guide/Inheritance_and_the_prototype_chain)
